1. Data Preparation
- Load the CSV file with metadata
- Create binary labels: consolidate health categories into "healthy" vs "unhealthy" (everything else)
- Load images from the bee_imgs/ folder based on filenames in CSV

2. Feature Extraction from Images
Random Forest can't work with raw images directly - you need to extract features first
Options:
- Hand-crafted features: Color histograms, texture features (HOG, LBP), edge features
- Pre-trained CNN features: Use a pre-trained model (ResNet, VGG, etc.) as a feature extractor, then feed those features to Random Forest
- Simple statistical features: Mean/std of RGB channels, image moments

3. Data Splitting
- Split into train/test sets (80/20 or similar)
- Ensure stratified split to maintain class balance
- Keep random state consistent for reproducibility

4. Check Class Distribution
- Analyze healthy vs unhealthy counts
- If imbalanced, plan handling (class weights, SMOTE, etc.)

5. Feature Scaling
- Normalize/standardize extracted features (Random Forest is less sensitive but still helps)

6. Model Training
- Build Random Forest classifier
- Consider hyperparameters: n_estimators, max_depth, min_samples_split
- Optionally use GridSearchCV or RandomizedSearchCV for tuning

7. Model Evaluation
- Accuracy, precision, recall, F1-score
- Confusion matrix
- ROC curve and AUC
- Feature importance analysis

8. Results Visualization
- Plot confusion matrix
- Feature importance bar chart
- Model performance metrics comparison

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

In [3]:
# Load the CSV file with metadata
bee_data = pd.read_csv("../data/bee_data.csv")


In [4]:
# Create binary labels: also merge health categories into "healthy" vs "unhealthy"
# Check unique health categories first
print("Original health categories:")
print(bee_data['health'].unique())
print("\nOriginal class distribution:")
print(bee_data['health'].value_counts())

# Create binary labels - everything except "healthy" becomes "unhealthy"
# lambda function: if the input in the "health" column of the dataset equals "healthy", it will be classified as that, if not, it will be "unhealthy" -> this is mandatory for the random forest, because it can only deal with binary classification
bee_data['binary_health'] = bee_data['health'].apply(lambda x: 'healthy' if x == 'healthy' else 'unhealthy')

print("\n" + "="*50)
print("Binary class distribution:")
print(bee_data['binary_health'].value_counts())

Original health categories:
['hive being robbed' 'healthy' 'few varrao, hive beetles' 'ant problems'
 'missing queen' 'Varroa, Small Hive Beetles']

Original class distribution:
health
healthy                       3384
few varrao, hive beetles       579
Varroa, Small Hive Beetles     472
ant problems                   457
hive being robbed              251
missing queen                   29
Name: count, dtype: int64

Binary class distribution:
binary_health
healthy      3384
unhealthy    1788
Name: count, dtype: int64


We can see that we have:
healthy      3384
unhealthy    1788
bees...
This will definitely benefit from undersampling the healthy class.

In [5]:
# Preparation of features (X) and binary target (y)
x = bee_data[['file']]  # We'll only need the filename to load images
y = bee_data['binary_health'] # binary target

# Split data into train (80%) and test (20%)
# Stratified split to maintain class balance -> ratio of healthy vs unhealthy (we can see the ratio at the bottom)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# Shape / # rows, # columns for the training and test sets
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Check binary class distribution in train and test sets
print("\n" + "="*50)
print("Training set class distribution:")
print(y_train.value_counts())

print("\nTest set class distribution:")
print(y_test.value_counts())

# Calculate percentages
print("\n" + "="*50)
print("Training set percentages:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest set percentages:")
print(y_test.value_counts(normalize=True) * 100)

x_train shape: (4137, 1)
y_train shape: (4137,)
x_test shape: (1035, 1)
y_test shape: (1035,)

Training set class distribution:
binary_health
healthy      2707
unhealthy    1430
Name: count, dtype: int64

Test set class distribution:
binary_health
healthy      677
unhealthy    358
Name: count, dtype: int64

Training set percentages:
binary_health
healthy      65.433889
unhealthy    34.566111
Name: proportion, dtype: float64

Test set percentages:
binary_health
healthy      65.410628
unhealthy    34.589372
Name: proportion, dtype: float64


we can see that because of stratification, the ratios of about 65.4 on training set and 34.6 on test set were maintained.

Now we could handle this with setting class weights undersampling majority class (heakthy) or undersampling the minority class (unhealthy).

In [ ]:
# When training the Random Forest later, use class_weight parameter
# This automatically penalizes misclassifications of the minority class (unhealthy)



class_weights = compute_class_weight('balanced', 
                                      classes=np.unique(y_train),
                                      y=y_train)

class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print("Class weights:")
print(class_weight_dict)

In [ ]:
# Load images from the bee_imgs/ folder based on filenames in CSV

# Function to load an image from the bee_imgs folder
def load_image(filename):
    img_path = os.path.join("../data/bee_imgs", filename) # join directory name with filename
    return np.array(Image.open(img_path)) # return Image pixel data as numpy array

# Load all images from the training set as a list
x_train_images = []
for i in range(len(x_train)):
    x_train_images.append(load_image(x_train.iloc[i, 0])) # get filename: 0-th column, i-th row
print(f"Loaded {len(x_train_images)} training images")

# Load all images from the test set
x_test_images = []
for i in range(len(x_test)):
    x_test_images.append(load_image(x_test.iloc[i, 0]))
print(f"Loaded {len(x_test_images)} test images")